# ConnectTel Telecom Customer Churn Prediction
## Notebook 03: Model Training, Cross-Validation & Controlled Hyperparameter Tuning

**Objective:**
- Train candidate machine-learning models (Logistic Regression, Random Forest, XGBoost).
- Perform 5-Fold Stratified Cross-Validation inside scikit-learn pipelines to prevent data leakage.
- Execute controlled, lightweight hyperparameter tuning using `GridSearchCV`.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath('..'))

from src.data_preprocessing import load_raw_data, clean_telco_data, get_feature_columns, build_preprocessing_pipeline
from src.feature_engineering import create_features
from src.train_models import get_baseline_models, evaluate_cv_models, tune_hyperparameters

print("[INFO] Environment setup complete.")

--- 
## 1. Load Data & Prepare Stratified Train/Test Sets

In [ ]:
raw_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df_raw = load_raw_data(raw_path)
df_clean = clean_telco_data(df_raw)
df_feat = create_features(df_clean)

X = df_feat.drop(columns=['Churn', 'customerID'], errors='ignore')
y = df_feat['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

num_cols, cat_cols = get_feature_columns(X, target_col='', id_col='')
preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

--- 
## 2. 5-Fold Stratified Cross-Validation (Baseline Models)

In [ ]:
baseline_models = get_baseline_models(random_state=42)
cv_results_df = evaluate_cv_models(baseline_models, X_train, y_train, preprocessor, cv_splits=5)

print("\n--- 5-Fold Cross-Validation Metric Results ---")
cv_results_df

--- 
## 3. Controlled Hyperparameter Tuning with GridSearchCV

In [ ]:
# Tune Random Forest Classifier
best_rf_pipeline, best_rf_params, best_rf_score = tune_hyperparameters('Random Forest', X_train, y_train, preprocessor)
print("Best RF Params:", best_rf_params)

In [ ]:
# Tune XGBoost Classifier
best_xgb_pipeline, best_xgb_params, best_xgb_score = tune_hyperparameters('XGBoost', X_train, y_train, preprocessor)
print("Best XGBoost Params:", best_xgb_params)

--- 
## 4. Hyperparameter Tuning Summary
- Record cross-validation metrics across models.
- Proceed to Notebook 04 for final test set evaluation, ROC curves, SHAP explainability, and artifact saving.